In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q PySastrawi nltk scikit-learn pandas

import os
import re
import string
import pandas as pd
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')

Mounted at /content/drive


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
folder_path = '/content/drive/MyDrive/Kuliah/SEMESTER 11/NLP'
file_path = os.path.join(folder_path, 'id-movie-review-sentimentanalysis.tsv')

# Load TSV dataset
df = pd.read_csv(file_path, sep='\t', encoding='utf-8', encoding_errors='replace')

# Keep relevant columns and encode target labels
df = df[['tweet', 'sentiment']].copy()
df['label'] = df['sentiment'].map({'negative': 0, 'positive': 1})

print("Dataset loaded successfully! Shape:", df.shape)
print("\nClass distribution:\n", df['sentiment'].value_counts())
print("\nSample rows:")
display(df.head())

Dataset loaded successfully! Shape: (500, 3)

Class distribution:
 sentiment
positive    250
negative    250
Name: count, dtype: int64

Sample rows:


,tweet,sentiment,label
0,Kolaborasi seru @KineriaMovie : Angga Dwimas S...,positive,1
1,Wah :)) Filosofi Kopi harus sama bagus nya ata...,positive,1
2,"RT@looohh : Rio Dewanto: Penonton Film ""Filos...",positive,1
3,oh .. jadi mau ada film baru yang berasal dari...,positive,1
4,"Bacaan sore tentang Filosofi Film eh, Kopi RT ...",positive,1


In [4]:
# 1. Informal Indonesian (slang/alay) normalization dictionary
slang_dict = {
    'ga': 'tidak', 'gak': 'tidak', 'gk': 'tidak', 'engga': 'tidak', 'enggak': 'tidak', 'nggak': 'tidak',
    'bgt': 'banget', 'bgtt': 'banget', 'dg': 'dengan', 'dgn': 'dengan',
    'yg': 'yang', 'utk': 'untuk', 'krn': 'karena', 'karna': 'karena',
    'tp': 'tapi', 'tpi': 'tapi', 'jd': 'jadi', 'jdi': 'jadi',
    'sdh': 'sudah', 'udah': 'sudah', 'blm': 'belum', 'lom': 'belum',
    'gua': 'saya', 'gue': 'saya', 'gw': 'saya', 'aku': 'saya',
    'lu': 'kamu', 'lo': 'kamu', 'loe': 'kamu',
    'kalo': 'kalau', 'kl': 'kalau', 'klw': 'kalau',
    'bikin': 'buat', 'nonton': 'tonton', 'nyesek': 'sesak',
    'anjir': 'anjing', 'sih': '', 'nih': '', 'deh': '', 'dong': '', 'nya': ''
}

# 2. Indonesian Stopwords (keeping negation words intact)
indo_stopwords = set(stopwords.words('indonesian'))
negation_words = {'tidak', 'bukan', 'jangan', 'belum', 'kurang', 'tanpa'}
indo_stopwords = indo_stopwords - negation_words

# 3. Initialize Sastrawi Stemmer
stemmer = StemmerFactory().create_stemmer()

def preprocess_tweet(text, use_stemming=True):
    # Lowercase
    text = text.lower()

    # Remove corrupted unicode replacement character ()
    text = text.replace('\ufffd', ' ')

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|pic\.twitter\.com\S+', ' ', text)

    # Remove RT tags and @mentions
    text = re.sub(r'\brt\b\s*@?\w*|@\w+', ' ', text)

    # Remove cinema/Path check-in tags like "(at cinema21)" or "(with @user at ...)"
    text = re.sub(r'\((at|with|w/)\s+[^)]+\)', ' ', text)

    # Remove '#' from hashtags while keeping the word
    text = re.sub(r'#(\w+)', r'\1', text)

    # Remove numbers and punctuation
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))

    # Collapse elongated characters (e.g., "kereeeen" -> "keren")
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Tokenize, normalize slang, and remove stopwords
    tokens = text.split()
    cleaned_tokens = []
    for word in tokens:
        word = slang_dict.get(word, word)
        if word and word not in indo_stopwords and len(word) > 1:
            cleaned_tokens.append(word)

    text = ' '.join(cleaned_tokens)

    # Apply Sastrawi stemming (set use_stemming=False if using IndoBERT)
    if use_stemming:
        text = stemmer.stem(text)

    return text

print("Preprocessing 500 tweets (takes ~15-30 seconds)...")
df['clean_tweet'] = df['tweet'].apply(lambda x: preprocess_tweet(x, use_stemming=True))
print("Done!\n")

# Compare raw vs. cleaned tweets
for i in range(5):
    print(f"[{df['sentiment'].iloc[i].upper()}]")
    print(f"RAW  : {df['tweet'].iloc[i]}")
    print(f"CLEAN: {df['clean_tweet'].iloc[i]}\n")

Preprocessing 500 tweets (takes ~15-30 seconds)...
Done!

[POSITIVE]
RAW  : Kolaborasi seru @KineriaMovie : Angga Dwimas Sasongko Siap Garap "FILOSOFI KOPI" http://www.kapanlagi.com/showbiz/film/indonesia/angga-dwimas-sasongko-siap-garap-filosofi-kopi-333b35.html � @FILM _Indonesia @anggiakharisma
CLEAN: kolaborasi seru angga dwimas sasongko garap filosofi kopi indonesia

[POSITIVE]
RAW  : Wah :)) Filosofi Kopi harus sama bagus nya atau lebih @MirLes : @anggasasongko @GlennFredly Anggia, Film Cahaya Dari Timur bagus! @kineforum
CLEAN: filosofi kopi bagus anggia film cahaya timur bagus

[POSITIVE]
RAW  : RT@looohh : Rio Dewanto: Penonton Film  "Filosofi Kopi The Movie" Bisa Dapat Pengetahuan Tentang Kopi http://dlvrit.com/83Dxkb .."
CLEAN: rio dewanto tonton film filosofi kopi the movie tahu kopi

[POSITIVE]
RAW  : oh .. jadi mau ada film baru yang berasal dari novel judul nya Filosofi Kopi . Pokok nya aku wajib nonton!
CLEAN: oh film asal novel judul filosofi kopi pokok wajib tonton

[

In [5]:
# Save cleaned dataset directly to your NLP folder on Google Drive
output_path = os.path.join(folder_path, 'preprocessed_movie_reviews.csv')
df.to_csv(output_path, index=False)
print(f"Saved preprocessed dataset to: {output_path}")

# Stratified 80/20 Train-Test Split (400 train, 100 test)
X = df['clean_tweet']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF Vectorization (unigrams + bigrams)
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=3000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("\nReady for modeling:")
print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape :", X_test_tfidf.shape)

Saved preprocessed dataset to: /content/drive/MyDrive/Kuliah/SEMESTER 11/NLP/preprocessed_movie_reviews.csv

Ready for modeling:
X_train_tfidf shape: (400, 3000)
X_test_tfidf shape : (100, 3000)
